In [4]:
black_dir='/content/drive/MyDrive/CP_DS/DS/black'
white_dir='/content/drive/MyDrive/CP_DS/DS/white'

In [5]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [7]:
img_height, img_width = 75, 75
batch_size = 16

white_ds = tf.keras.utils.image_dataset_from_directory(
    white_dir,
    labels='inferred',
    label_mode='int',
    image_size=(img_height, img_width),
    interpolation='nearest',
    batch_size=batch_size,
    shuffle=True
)
img_height, img_width = 75, 75
batch_size = 16

black_ds = tf.keras.utils.image_dataset_from_directory(
    black_dir,
    labels='inferred',
    label_mode='int',
    image_size=(img_height, img_width),
    interpolation='nearest',
    batch_size=batch_size,
    shuffle=True
)

Found 1432 files belonging to 1 classes.
Found 948 files belonging to 1 classes.


In [8]:
# Define the data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.3),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.2, width_factor=0.2),
    tf.keras.layers.RandomShear(x_factor=0.2,y_factor=0.2)
])

# Apply augmentation
augmented_images_white=[]
label_white=[]
for i in range(3):
  for image_batch,label_batch in white_ds:
    augmented_images_white.append(data_augmentation(image_batch))
    label_white.append(label_batch)
ds1=white_ds.take(5)
for image_batch,label_batch in ds1:
    augmented_images_white.append(data_augmentation(image_batch))
    label_white.append(label_batch)
aug_images_white=np.concatenate(augmented_images_white)
aug_label_white=np.concatenate(label_white)
#
augmented_images_black=[]
label_black=[]
for i in range(4):
  for image_batch,label_batch in black_ds:
    augmented_images_black.append(data_augmentation(image_batch))
    label_black.append(label_batch)
ds1=black_ds.take(28)
for image_batch,label_batch in ds1:
    augmented_images_black.append(data_augmentation(image_batch))
    label_black.append(label_batch)
aug_images_black=np.concatenate(augmented_images_black)
aug_label_black=np.concatenate(label_black)
print(aug_images_black.shape)
print(aug_label_black.shape)

(4240, 75, 75, 3)
(4240,)


In [9]:
data_aug_input=np.concatenate((aug_images_white,aug_images_black))
data_aug_input.shape

(8616, 75, 75, 3)

In [13]:
base_model_densenet=tf.keras.applications.densenet.DenseNet121(include_top=False,
                                                               input_shape=(75,75,3))
base_model_densenet.trainable=False

In [15]:
inputs=tf.keras.Input(shape=(75,75,3)) # input layer x=data_aug(inputs)
x=tf.keras.applications.densenet.preprocess_input(inputs)
x=base_model_densenet(x,training=False)
outputs=tf.keras.layers.GlobalAveragePooling2D()(x)
model_densenet=tf.keras.Model(inputs,outputs)
model_densenet.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add (Add)                       │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 2, 2, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,037,504 (26.85 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 7,037,504 (26.85 MB)

In [16]:
feature_densenet=model_densenet.predict(data_aug_input)

270/270 ━━━━━━━━━━━━━━━━━━━━ 157s 566ms/step


In [17]:
feature_densenet.shape

(8616, 1024)

In [18]:
import pandas as pd
df=pd.DataFrame(feature_densenet)
df.to_csv('/content/drive/MyDrive/CP_DS/DS/feature_densenet.csv',index=False)
print('Feature Written')

Feature Written


In [19]:
base_model_resnet=tf.keras.applications.resnet.ResNet50(include_top=False,
                                                               input_shape=(75,75,3))
base_model_resnet.trainable=False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [21]:
inputs=tf.keras.Input(shape=(75,75,3)) # input layer x=data_aug(inputs)
x=tf.keras.applications.resnet.preprocess_input(inputs)
x=base_model_resnet(x,training=False)
outputs=tf.keras.layers.GlobalAveragePooling2D()(x)
model_resnet=tf.keras.Model(inputs,outputs)
model_resnet.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 75, 75, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 75, 75)    │          0 │ input_layer_7[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 75, 75)    │          0 │ input_layer_7[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 75, 75)    │          0 │ input_layer_7[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 75, 75, 3) │          0 │ get_item_3[0][0], │
│                     │                   │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 75, 75, 3) │          0 │ stack_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 3, 3,      │ 23,587,712 │ add_2[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[1][0]    │
│ (GlobalAveragePool… │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [22]:
feature_resnet=model_resnet.predict(data_aug_input)

270/270 ━━━━━━━━━━━━━━━━━━━━ 222s 814ms/step


In [23]:
print(feature_resnet.shape)
pd.DataFrame(feature_resnet).to_csv('/content/drive/MyDrive/CP_DS/DS/feature_resnet.csv',index=False)
print('Feature Written')


(8616, 2048)
Feature Written


In [24]:
base_model_inception=tf.keras.applications.inception_v3.InceptionV3(include_top=False,
                                                               input_shape=(75,75,3))
base_model_inception.trainable=False

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [25]:
inputs=tf.keras.Input(shape=(75,75,3)) # input layer x=data_aug(inputs)
x=tf.keras.applications.inception_v3.preprocess_input(inputs)
x=base_model_inception(x,training=False)
outputs=tf.keras.layers.GlobalAveragePooling2D()(x)
model_inception=tf.keras.Model(inputs,outputs)
model_inception.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_2 (TrueDivide)      │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 1, 1, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,802,784 (83.17 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 21,802,784 (83.17 MB)

In [26]:
feature_inception=model_inception.predict(data_aug_input)

270/270 ━━━━━━━━━━━━━━━━━━━━ 94s 340ms/step


In [27]:
print(feature_inception.shape)
pd.DataFrame(feature_inception).to_csv('/content/drive/MyDrive/CP_DS/DS/feature_inception.csv',index=False)
print('Feature Written')


(8616, 2048)
Feature Written


In [28]:
base_model_xception=tf.keras.applications.xception.Xception(include_top=False,
                                                               input_shape=(75,75,3))
base_model_xception.trainable=False

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [29]:
inputs=tf.keras.Input(shape=(75,75,3)) # input layer x=data_aug(inputs)
x=tf.keras.applications.xception.preprocess_input(inputs)
x=base_model_xception(x,training=False)
outputs=tf.keras.layers.GlobalAveragePooling2D()(x)
model_xception=tf.keras.Model(inputs,outputs)
model_xception.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_3 (TrueDivide)      │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ xception (Functional)           │ (None, 3, 3, 2048)     │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,861,480 (79.58 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 20,861,480 (79.58 MB)

In [30]:
feature_xception=model_xception.predict(data_aug_input)

270/270 ━━━━━━━━━━━━━━━━━━━━ 240s 886ms/step


In [31]:
print(feature_xception.shape)
pd.DataFrame(feature_xception).to_csv('/content/drive/MyDrive/CP_DS/DS/feature_xception.csv',index=False)
print('Feature Written')


(8616, 2048)
Feature Written


In [32]:
base_model_vgg19=tf.keras.applications.vgg19.VGG19(include_top=False,
                                                               input_shape=(75,75,3))
base_model_vgg19.trainable=False

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [33]:
inputs=tf.keras.Input(shape=(75,75,3)) # input layer x=data_aug(inputs)
x=tf.keras.applications.vgg19.preprocess_input(inputs)
x=base_model_vgg19(x,training=False)
outputs=tf.keras.layers.GlobalAveragePooling2D()(x)
model_vgg19=tf.keras.Model(inputs,outputs)
model_vgg19.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 75, 75, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_6          │ (None, 75, 75)    │          0 │ input_layer_13[0… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_7          │ (None, 75, 75)    │          0 │ input_layer_13[0… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_8          │ (None, 75, 75)    │          0 │ input_layer_13[0… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_2 (Stack)     │ (None, 75, 75, 3) │          0 │ get_item_6[0][0], │
│                     │                   │            │ get_item_7[0][0], │
│                     │                   │            │ get_item_8[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_15 (Add)        │ (None, 75, 75, 3) │          0 │ stack_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg19 (Functional)  │ (None, 2, 2, 512) │ 20,024,384 │ add_15[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg19[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,024,384 (76.39 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 20,024,384 (76.39 MB)

In [34]:
feature_vgg19=model_vgg19.predict(data_aug_input)

270/270 ━━━━━━━━━━━━━━━━━━━━ 684s 3s/step


In [35]:
print(feature_vgg19.shape)
pd.DataFrame(feature_vgg19).to_csv('/content/drive/MyDrive/CP_DS/DS/feature_vgg19.csv',index=False)
print('Feature Written')


(8616, 512)
Feature Written
